In [0]:

from common_utils.bronze import read_raw, bronze_ingestor
from common_utils.logging import get_logger

logger = get_logger("Ds2b")

sources = {
    "S3_bucket": {
        "raw_folder": "S3_bucket",
        "file_format": "parquet",
        "read_options": {},
        "target_table": "retail_data_platform.bronze.S3_sales",
        "mode": "overwrite"
    },

    "Cosmosdb_sales_order": {
        "raw_folder": "Cosmosdb_sales_order",
        "file_format": "json",
        "read_options": {},
        "target_table": "retail_data_platform.bronze.Cosmosdb_sales_order",
        "mode": "overwrite"
    },

    "sqlserver_customers": {
        "raw_folder": "sqlserver_customers",
        "file_format": "csv",
        "read_options": {
            "header": "true",
            "inferSchema": "true"
        },
        "target_table": "retail_data_platform.bronze.sqlserver_customers",
        "mode": "overwrite"
    }
}

for source_name, config in sources.items():

    raw_path = f"/Volumes/retail_data_platform/bronze/raw_data/{config['raw_folder']}/2026-10-28/"

    logger.info("[%s] Reading from %s", source_name, raw_path)

    df = read_raw(
        spark,
        raw_path,
        file_format=config["file_format"],
        options=config["read_options"]
    )

    rows = bronze_ingestor(
        df,
        mode=config["mode"],
        target_table=config["target_table"]
    )

    logger.info("[%s] Ingested %s rows", source_name, rows)

logger.info("Bronze completed for all sources")
